# Упражнение: Один шаг TPE вручную (идея Optuna и Hyperopt)

## Задание

В `input.csv` — история 30 проб случайного поиска гиперпараметров бустинга: `log_lr` = log₁₀(learning_rate)
из [−3, 0], `subsample` из [0.5, 1] и ошибка на валидации `loss` (меньше — лучше).
В `candidates.csv` — 24 кандидата, из которых TPE должен выбрать следующую пробу.

Сделайте **один шаг TPE** (упрощённого, с фиксированной шириной ядер):

1. «Хорошие» пробы — **6** с наименьшим `loss` (γ = 20%), «плохие» — остальные 24.
2. Для каждого параметра отдельно постройте две оценки плотности Парзена — смесь гауссиан с центрами в значениях
   проб и одинаковой шириной: l(x) = (1/n_good)·Σ N(x; xᵢ, h²) по хорошим, g(x) — так же по плохим.
   Ширина h = 0.1 × длина диапазона: **0.3** для `log_lr`, **0.05** для `subsample`.
3. Совместные плотности — произведения по параметрам: l(x) = l_lr(x)·l_sub(x), g(x) = g_lr(x)·g_sub(x).
4. Для каждого кандидата посчитайте `log_l = ln l(x)`, `log_g = ln g(x)`, `log_ratio = log_l − log_g`
   и `rank` — место по убыванию log_ratio (1 — следующая проба TPE).

## Подсказки

- TPE ищет место, где хорошие пробы плотные, а плохие — редкие: максимум l(x)/g(x) эквивалентен максимуму EI.
- Плотность одной смеси для всех кандидатов сразу: `norm.pdf(cand[:, None], loc=centers[None, :], scale=h).mean(axis=1)`.
- Хорошие пробы: `df.nsmallest(6, 'loss')`, плохие — все остальные. Логарифм произведения — сумма логарифмов;
  rank: `(-log_ratio).argsort().argsort() + 1`.

## Ожидаемый формат выходного файла

CSV с колонками `candidate, log_l, log_g, log_ratio, rank` — 24 строки в порядке `candidates.csv`.
Логарифмы — натуральные, округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/optuna_hyperopt
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.